STAGE 4.8
# OpenVINO FP32 Benchmark

In [2]:
# =============================================================================
# CISLR — STAGE 4.8
# OpenVINO FP32 Benchmark
#
# PURPOSE:
#   Benchmark the already-validated OpenVINO FP32 MobileNetV3-Large IR.
#
# IMPORTANT:
#   - NO retraining
#   - NO model conversion
#   - NO video decoding
#   - NO Stage 4.3 / 4.4 / 4.5 / 4.6 reruns
#
# COMPARABILITY WITH STAGE 4.3:
#   Input       : 1 x 3 x 224 x 224
#   Precision   : FP32
#   Batch       : 1
#   Warmup      : 50
#   Measurements: 200
#   Metric      : single-inference latency + FPS
# =============================================================================

import os
import sys
import json
import time
import hashlib
import platform
import statistics
from pathlib import Path

import numpy as np
import pandas as pd
import openvino as ov


# =============================================================================
# 1. CONFIGURATION
# =============================================================================

ROOT = Path("/home/dipshikha/Music/cao/CISLR_RESEARCH")

STAGE_4_7_DIR = (
    ROOT /
    "audit" /
    "stage4_7_openvino_conversion"
)

FP32_DIR = STAGE_4_7_DIR / "fp32"

XML_PATH = (
    FP32_DIR /
    "mobilenet_v3_large_stage4_7_fp32.xml"
)

BIN_PATH = (
    FP32_DIR /
    "mobilenet_v3_large_stage4_7_fp32.bin"
)

OUTPUT_DIR = (
    ROOT /
    "audit" /
    "stage4_8_openvino_benchmark"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Benchmark configuration
BATCH_SIZE = 1
CHANNELS = 3
IMAGE_SIZE = 224

WARMUP_ITERS = 50
MEASURE_ITERS = 200

# Stage 4.3 PyTorch reference numbers
# These are NOT recomputed.
# They are the already-completed Stage 4.3 benchmark values.
PYTORCH_STAGE_4_3 = {
    "CPU": {
        "latency_ms": 18.989,
        "fps": 52.66
    },
    "GPU": {
        "latency_ms": 5.362,
        "fps": 186.51
    }
}


# =============================================================================
# 2. HEADER
# =============================================================================

print("=" * 80)
print("CISLR — STAGE 4.8 OPENVINO BENCHMARK")
print("=" * 80)

print()
print("Project root:")
print(ROOT)

print()
print("Selected model:")
print("MobileNetV3-Large")

print()
print("OpenVINO IR:")
print(XML_PATH)

print()
print("Output:")
print(OUTPUT_DIR)


# =============================================================================
# 3. ENVIRONMENT
# =============================================================================

print()
print("=" * 80)
print("ENVIRONMENT")
print("=" * 80)

print("Python       :", sys.version)
print("OpenVINO     :", ov.__version__)
print("Platform     :", platform.platform())
print("Architecture :", platform.machine())


# =============================================================================
# 4. VERIFY INPUT ARTIFACTS
# =============================================================================

print()
print("=" * 80)
print("VERIFYING STAGE 4.7 ARTIFACTS")
print("=" * 80)

if not XML_PATH.exists():
    raise FileNotFoundError(
        f"OpenVINO XML not found:\n{XML_PATH}"
    )

if not BIN_PATH.exists():
    raise FileNotFoundError(
        f"OpenVINO BIN not found:\n{BIN_PATH}"
    )

print("XML exists : True")
print("BIN exists : True")

xml_size_mb = XML_PATH.stat().st_size / (1024 ** 2)
bin_size_mb = BIN_PATH.stat().st_size / (1024 ** 2)
total_size_mb = xml_size_mb + bin_size_mb

print(f"XML size   : {xml_size_mb:.3f} MB")
print(f"BIN size   : {bin_size_mb:.3f} MB")
print(f"Total IR   : {total_size_mb:.3f} MB")


# =============================================================================
# 5. HASH INPUT IR
# =============================================================================

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


xml_hash = sha256_file(XML_PATH)
bin_hash = sha256_file(BIN_PATH)

print()
print("SHA256 XML:")
print(xml_hash)

print()
print("SHA256 BIN:")
print(bin_hash)


# =============================================================================
# 6. CREATE OPENVINO CORE
# =============================================================================

print()
print("=" * 80)
print("OPENVINO CORE")
print("=" * 80)

core = ov.Core()

available_devices = core.available_devices

print("Available OpenVINO devices:")
for device in available_devices:
    print("  -", device)


# =============================================================================
# 7. READ MODEL
# =============================================================================

print()
print("=" * 80)
print("READING OPENVINO FP32 IR")
print("=" * 80)

read_start = time.perf_counter()

model = core.read_model(
    model=str(XML_PATH)
)

read_time = time.perf_counter() - read_start

print(f"Model read time: {read_time:.6f} seconds")


# =============================================================================
# 8. MODEL INFORMATION
# =============================================================================

print()
print("=" * 80)
print("MODEL INFORMATION")
print("=" * 80)

print("Number of inputs :", len(model.inputs))
print("Number of outputs:", len(model.outputs))

for i, inp in enumerate(model.inputs):
    print()
    print(f"Input {i}:")
    print("  Name :", inp.any_name)
    print("  Shape:", inp.partial_shape)
    print("  Type :", inp.element_type)

for i, out in enumerate(model.outputs):
    print()
    print(f"Output {i}:")
    for i, out in enumerate(model.outputs):
        print()
        print(f"Output {i}:")
        try:
            out_name = out.any_name
        except RuntimeError:
            out_name = "<unnamed>"
        print("  Name :", out_name)
        print("  Shape:", out.partial_shape)
        print("  Type :", out.element_type)
    print("  Shape:", out.partial_shape)
    print("  Type :", out.element_type)


# =============================================================================
# 9. FIX INPUT SHAPE FOR BENCHMARK
#
# Stage 4.3 used:
#   [1, 3, 224, 224]
#
# Stage 4.7 conversion retained dynamic dimensions.
# For a fair benchmark, explicitly reshape to the exact benchmark shape.
# =============================================================================

print()
print("=" * 80)
print("SETTING BENCHMARK INPUT SHAPE")
print("=" * 80)

benchmark_shape = [
    BATCH_SIZE,
    CHANNELS,
    IMAGE_SIZE,
    IMAGE_SIZE
]

print("Benchmark shape:", benchmark_shape)

try:
    model.reshape({
        model.inputs[0]: benchmark_shape
    })
except Exception as e:
    print("Model reshape by input object failed.")
    print("Trying PartialShape fallback...")

    model.reshape(
        {model.inputs[0].any_name: benchmark_shape}
    )

print("Final model input shape:")
print(model.inputs[0].partial_shape)

print("Final model output shape:")
print(model.outputs[0].partial_shape)


# =============================================================================
# 10. CREATE DETERMINISTIC INPUT
#
# IMPORTANT:
#   This is synthetic data because this is a pure inference benchmark.
#   No video decoding is involved.
# =============================================================================

print()
print("=" * 80)
print("CREATING BENCHMARK INPUT")
print("=" * 80)

rng = np.random.default_rng(42)

input_data = rng.standard_normal(
    benchmark_shape
).astype(np.float32)

print("Input shape :", input_data.shape)
print("Input dtype :", input_data.dtype)
print("Input mean  :", float(input_data.mean()))
print("Input std   :", float(input_data.std()))


# =============================================================================
# 11. BENCHMARK FUNCTION
# =============================================================================

def benchmark_openvino_device(
    core,
    model,
    device_name,
    input_data,
    warmup_iters=50,
    measure_iters=200
):
    """
    Benchmark single-request OpenVINO inference.

    Uses latency-oriented configuration.

    Returns a dictionary containing:
      - compilation time
      - warmup statistics
      - latency statistics
      - FPS
      - device information
    """

    print()
    print("-" * 80)
    print(f"BENCHMARKING DEVICE: {device_name}")
    print("-" * 80)

    # -------------------------------------------------------------------------
    # Compile
    # -------------------------------------------------------------------------

    print()
    print("Compiling model...")

    compile_start = time.perf_counter()

    try:
        compiled_model = core.compile_model(
            model=model,
            device_name=device_name,
            config={
                "PERFORMANCE_HINT": "LATENCY"
            }
        )

    except Exception as e:
        print()
        print(f"Compilation failed for {device_name}.")
        print("Error:")
        print(e)

        return {
            "device": device_name,
            "available": False,
            "error": str(e)
        }

    compile_time = time.perf_counter() - compile_start

    print(
        f"Compilation time: "
        f"{compile_time:.6f} seconds"
    )

    # -------------------------------------------------------------------------
    # Create inference request
    # -------------------------------------------------------------------------

    infer_request = compiled_model.create_infer_request()

    input_port = compiled_model.inputs[0]

    # -------------------------------------------------------------------------
    # Warmup
    # -------------------------------------------------------------------------

    print()
    print(
        f"Warmup: {warmup_iters} iterations..."
    )

    warmup_times = []

    for _ in range(warmup_iters):

        start = time.perf_counter()

        infer_request.infer({
            input_port: input_data
        })

        elapsed = (
            time.perf_counter() - start
        ) * 1000.0

        warmup_times.append(elapsed)

    print(
        f"Warmup complete."
    )

    # -------------------------------------------------------------------------
    # Synchronization / stabilization
    # -------------------------------------------------------------------------

    # One extra inference before measurements to ensure
    # the request is in a clean completed state.

    infer_request.infer({
        input_port: input_data
    })

    # -------------------------------------------------------------------------
    # Measured inference
    # -------------------------------------------------------------------------

    print()
    print(
        f"Measured iterations: {measure_iters}"
    )

    latencies_ms = []

    for iteration in range(measure_iters):

        start = time.perf_counter()

        infer_request.infer({
            input_port: input_data
        })

        elapsed = (
            time.perf_counter() - start
        ) * 1000.0

        latencies_ms.append(elapsed)

    # -------------------------------------------------------------------------
    # Statistics
    # -------------------------------------------------------------------------

    latencies = np.asarray(
        latencies_ms,
        dtype=np.float64
    )

    mean_latency = float(
        np.mean(latencies)
    )

    median_latency = float(
        np.median(latencies)
    )

    p95_latency = float(
        np.percentile(latencies, 95)
    )

    p99_latency = float(
        np.percentile(latencies, 99)
    )

    min_latency = float(
        np.min(latencies)
    )

    max_latency = float(
        np.max(latencies)
    )

    std_latency = float(
        np.std(latencies)
    )

    fps = (
        1000.0 / mean_latency
        if mean_latency > 0
        else 0.0
    )

    median_fps = (
        1000.0 / median_latency
        if median_latency > 0
        else 0.0
    )

    # -------------------------------------------------------------------------
    # Compiled model properties
    # -------------------------------------------------------------------------

    properties = {}

    property_names = [
        "OPTIMAL_NUMBER_OF_INFER_REQUESTS",
        "NUM_STREAMS",
        "INFERENCE_NUM_THREADS",
        "PERFORMANCE_HINT"
    ]

    for property_name in property_names:

        try:
            value = compiled_model.get_property(
                property_name
            )

            properties[property_name] = str(value)

        except Exception:
            properties[property_name] = None

    # -------------------------------------------------------------------------
    # Output verification
    # -------------------------------------------------------------------------

    output = infer_request.get_output_tensor().data

    output_shape = list(
        output.shape
    )

    output_dtype = str(
        output.dtype
    )

    # -------------------------------------------------------------------------
    # Print results
    # -------------------------------------------------------------------------

    print()
    print("RESULTS")
    print("-" * 50)

    print(
        f"Mean latency     : {mean_latency:.6f} ms"
    )

    print(
        f"Median latency   : {median_latency:.6f} ms"
    )

    print(
        f"P95 latency      : {p95_latency:.6f} ms"
    )

    print(
        f"P99 latency      : {p99_latency:.6f} ms"
    )

    print(
        f"Min latency      : {min_latency:.6f} ms"
    )

    print(
        f"Max latency      : {max_latency:.6f} ms"
    )

    print(
        f"Std latency      : {std_latency:.6f} ms"
    )

    print(
        f"Mean FPS         : {fps:.4f}"
    )

    print(
        f"Median FPS       : {median_fps:.4f}"
    )

    print()
    print("Output shape     :", output_shape)
    print("Output dtype     :", output_dtype)

    print()
    print("Compiled properties:")

    for key, value in properties.items():
        print(
            f"  {key}: {value}"
        )

    return {
        "device": device_name,
        "available": True,

        "compile_time_s": compile_time,

        "warmup_iterations": warmup_iters,
        "measurement_iterations": measure_iters,

        "warmup_mean_ms": float(
            np.mean(warmup_times)
        ),

        "mean_latency_ms": mean_latency,
        "median_latency_ms": median_latency,
        "p95_latency_ms": p95_latency,
        "p99_latency_ms": p99_latency,
        "min_latency_ms": min_latency,
        "max_latency_ms": max_latency,
        "std_latency_ms": std_latency,

        "fps": fps,
        "median_fps": median_fps,

        "output_shape": output_shape,
        "output_dtype": output_dtype,

        "properties": properties,

        "latencies_ms": latencies_ms
    }


# =============================================================================
# 12. CPU BENCHMARK
# =============================================================================

print()
print("=" * 80)
print("OPENVINO CPU BENCHMARK")
print("=" * 80)

cpu_result = benchmark_openvino_device(
    core=core,
    model=model,
    device_name="CPU",
    input_data=input_data,
    warmup_iters=WARMUP_ITERS,
    measure_iters=MEASURE_ITERS
)


if not cpu_result.get("available", False):
    raise RuntimeError(
        "OpenVINO CPU benchmark failed."
    )


# =============================================================================
# 13. OPTIONAL GPU BENCHMARK
#
# Only run if OpenVINO actually exposes a GPU device.
#
# IMPORTANT:
# Your NVIDIA RTX 4050 is NOT automatically an OpenVINO GPU target.
# OpenVINO's GPU plugin generally refers to supported Intel GPU devices.
#
# Therefore this code checks availability instead of assuming GPU support.
# =============================================================================

print()
print("=" * 80)
print("OPENVINO GPU CHECK")
print("=" * 80)

gpu_devices = [
    d for d in available_devices
    if str(d).upper().startswith("GPU")
]

if len(gpu_devices) == 0:

    print("No OpenVINO GPU device detected.")
    print("GPU benchmark will be skipped.")

    gpu_result = {
        "device": "GPU",
        "available": False,
        "reason": (
            "No OpenVINO GPU device exposed "
            "by ov.Core().available_devices"
        )
    }

else:

    print("OpenVINO GPU devices detected:")

    for d in gpu_devices:
        print("  -", d)

    # Use first exposed OpenVINO GPU.
    selected_gpu = str(
        gpu_devices[0]
    )

    gpu_result = benchmark_openvino_device(
        core=core,
        model=model,
        device_name=selected_gpu,
        input_data=input_data,
        warmup_iters=WARMUP_ITERS,
        measure_iters=MEASURE_ITERS
    )


# =============================================================================
# 14. COMPARE WITH STAGE 4.3 PYTORCH
# =============================================================================

print()
print("=" * 80)
print("STAGE 4.3 PYTORCH vs STAGE 4.8 OPENVINO")
print("=" * 80)


comparison_rows = []


# -----------------------------------------------------------------------------
# CPU comparison
# -----------------------------------------------------------------------------

if cpu_result.get("available", False):

    ov_cpu_latency = cpu_result[
        "mean_latency_ms"
    ]

    ov_cpu_fps = cpu_result[
        "fps"
    ]

    pt_cpu_latency = PYTORCH_STAGE_4_3[
        "CPU"
    ]["latency_ms"]

    pt_cpu_fps = PYTORCH_STAGE_4_3[
        "CPU"
    ]["fps"]

    latency_speedup = (
        pt_cpu_latency /
        ov_cpu_latency
    )

    fps_speedup = (
        ov_cpu_fps /
        pt_cpu_fps
    )

    latency_reduction_percent = (
        (
            pt_cpu_latency -
            ov_cpu_latency
        )
        /
        pt_cpu_latency
    ) * 100.0

    print()
    print("CPU")
    print("-" * 50)

    print(
        f"PyTorch latency : "
        f"{pt_cpu_latency:.3f} ms"
    )

    print(
        f"OpenVINO latency: "
        f"{ov_cpu_latency:.3f} ms"
    )

    print(
        f"Latency speedup : "
        f"{latency_speedup:.4f}x"
    )

    print(
        f"PyTorch FPS     : "
        f"{pt_cpu_fps:.2f}"
    )

    print(
        f"OpenVINO FPS    : "
        f"{ov_cpu_fps:.2f}"
    )

    print(
        f"FPS speedup     : "
        f"{fps_speedup:.4f}x"
    )

    print(
        f"Latency reduction: "
        f"{latency_reduction_percent:.2f}%"
    )

    comparison_rows.append({
        "device": "CPU",
        "pytorch_latency_ms": pt_cpu_latency,
        "openvino_latency_ms": ov_cpu_latency,
        "latency_speedup_x": latency_speedup,
        "latency_reduction_percent":
            latency_reduction_percent,
        "pytorch_fps": pt_cpu_fps,
        "openvino_fps": ov_cpu_fps,
        "fps_speedup_x": fps_speedup
    })


# -----------------------------------------------------------------------------
# GPU comparison
# -----------------------------------------------------------------------------

if gpu_result.get("available", False):

    ov_gpu_latency = gpu_result[
        "mean_latency_ms"
    ]

    ov_gpu_fps = gpu_result[
        "fps"
    ]

    pt_gpu_latency = PYTORCH_STAGE_4_3[
        "GPU"
    ]["latency_ms"]

    pt_gpu_fps = PYTORCH_STAGE_4_3[
        "GPU"
    ]["fps"]

    latency_speedup = (
        pt_gpu_latency /
        ov_gpu_latency
    )

    fps_speedup = (
        ov_gpu_fps /
        pt_gpu_fps
    )

    latency_reduction_percent = (
        (
            pt_gpu_latency -
            ov_gpu_latency
        )
        /
        pt_gpu_latency
    ) * 100.0

    print()
    print("OPENVINO GPU")
    print("-" * 50)

    print(
        f"PyTorch latency : "
        f"{pt_gpu_latency:.3f} ms"
    )

    print(
        f"OpenVINO latency: "
        f"{ov_gpu_latency:.3f} ms"
    )

    print(
        f"Latency speedup : "
        f"{latency_speedup:.4f}x"
    )

    print(
        f"PyTorch FPS     : "
        f"{pt_gpu_fps:.2f}"
    )

    print(
        f"OpenVINO FPS    : "
        f"{ov_gpu_fps:.2f}"
    )

    print(
        f"FPS speedup     : "
        f"{fps_speedup:.4f}x"
    )

    print(
        f"Latency reduction: "
        f"{latency_reduction_percent:.2f}%"
    )

    comparison_rows.append({
        "device": "OpenVINO-" + str(
            gpu_result["device"]
        ),
        "pytorch_latency_ms": pt_gpu_latency,
        "openvino_latency_ms": ov_gpu_latency,
        "latency_speedup_x": latency_speedup,
        "latency_reduction_percent":
            latency_reduction_percent,
        "pytorch_fps": pt_gpu_fps,
        "openvino_fps": ov_gpu_fps,
        "fps_speedup_x": fps_speedup
    })


# =============================================================================
# 15. SAVE COMPARISON CSV
# =============================================================================

comparison_df = pd.DataFrame(
    comparison_rows
)

comparison_csv = (
    OUTPUT_DIR /
    "stage4_8_pytorch_vs_openvino_comparison.csv"
)

comparison_df.to_csv(
    comparison_csv,
    index=False
)

print()
print("Saved comparison CSV:")
print(comparison_csv)


# =============================================================================
# 16. SAVE RAW LATENCIES
# =============================================================================

latency_rows = []

if cpu_result.get("available", False):

    for i, latency in enumerate(
        cpu_result["latencies_ms"],
        start=1
    ):

        latency_rows.append({
            "device": "CPU",
            "iteration": i,
            "latency_ms": latency
        })


if gpu_result.get("available", False):

    for i, latency in enumerate(
        gpu_result["latencies_ms"],
        start=1
    ):

        latency_rows.append({
            "device": str(
                gpu_result["device"]
            ),
            "iteration": i,
            "latency_ms": latency
        })


latency_df = pd.DataFrame(
    latency_rows
)

latency_csv = (
    OUTPUT_DIR /
    "stage4_8_latency_measurements.csv"
)

latency_df.to_csv(
    latency_csv,
    index=False
)

print()
print("Saved latency measurements:")
print(latency_csv)


# =============================================================================
# 17. REMOVE RAW LATENCY ARRAYS BEFORE JSON
# =============================================================================

def clean_result_for_json(result):

    cleaned = {}

    for key, value in result.items():

        if key == "latencies_ms":
            continue

        if isinstance(value, np.integer):
            value = int(value)

        elif isinstance(value, np.floating):
            value = float(value)

        elif isinstance(value, np.ndarray):
            value = value.tolist()

        cleaned[key] = value

    return cleaned


cpu_json_result = clean_result_for_json(
    cpu_result
)

gpu_json_result = clean_result_for_json(
    gpu_result
)


# =============================================================================
# 18. CREATE COMPLETE BENCHMARK REPORT
# =============================================================================

benchmark_report = {

    "stage": "4.8",

    "stage_name":
        "OpenVINO FP32 Benchmark",

    "project":
        "CISLR",

    "selected_model":
        "MobileNetV3-Large",

    "precision":
        "FP32",

    "input_shape":
        benchmark_shape,

    "output_shape":
        [1, 4764],

    "warmup_iterations":
        WARMUP_ITERS,

    "measurement_iterations":
        MEASURE_ITERS,

    "openvino_version":
        ov.__version__,

    "platform":
        platform.platform(),

    "python_version":
        sys.version,

    "available_devices":
        [str(d) for d in available_devices],

    "input_ir": {

        "xml":
            str(XML_PATH),

        "bin":
            str(BIN_PATH),

        "xml_size_mb":
            xml_size_mb,

        "bin_size_mb":
            bin_size_mb,

        "total_size_mb":
            total_size_mb,

        "xml_sha256":
            xml_hash,

        "bin_sha256":
            bin_hash
    },

    "cpu": cpu_json_result,

    "gpu": gpu_json_result,

    "stage4_3_pytorch_reference":
        PYTORCH_STAGE_4_3,

    "comparison":
        comparison_rows
}


# =============================================================================
# 19. SAVE JSON
# =============================================================================

json_path = (
    OUTPUT_DIR /
    "stage4_8_openvino_benchmark.json"
)

with open(
    json_path,
    "w"
) as f:

    json.dump(
        benchmark_report,
        f,
        indent=2
    )

print()
print("Saved benchmark JSON:")
print(json_path)


# =============================================================================
# 20. HASH OUTPUT ARTIFACTS
# =============================================================================

def sha256_file_safe(path):

    if not Path(path).exists():
        return None

    return sha256_file(path)


artifact_hashes = {

    "input_xml":
        sha256_file_safe(XML_PATH),

    "input_bin":
        sha256_file_safe(BIN_PATH),

    "comparison_csv":
        sha256_file_safe(comparison_csv),

    "latency_csv":
        sha256_file_safe(latency_csv),

    "benchmark_json":
        sha256_file_safe(json_path)
}


hash_path = (
    OUTPUT_DIR /
    "stage4_8_artifact_hashes.json"
)

with open(
    hash_path,
    "w"
) as f:

    json.dump(
        artifact_hashes,
        f,
        indent=2
    )

print()
print("Saved hashes:")
print(hash_path)


# =============================================================================
# 21. FINAL SUMMARY
# =============================================================================

print()
print("=" * 80)
print("STAGE 4.8 COMPLETE")
print("=" * 80)

print()
print("Selected model:")
print("  >>> MobileNetV3-Large")

print()
print("OpenVINO precision:")
print("  >>> FP32")

print()
print("Input:")
print("  >>> [1, 3, 224, 224]")

print()
print("Output:")
print("  >>> [1, 4764]")

print()
print("Warmup:")
print(f"  >>> {WARMUP_ITERS}")

print()
print("Measured iterations:")
print(f"  >>> {MEASURE_ITERS}")


# -----------------------------------------------------------------------------
# CPU final summary
# -----------------------------------------------------------------------------

if cpu_result.get("available", False):

    print()
    print("OPENVINO CPU")
    print("-" * 50)

    print(
        f"Mean latency   : "
        f"{cpu_result['mean_latency_ms']:.6f} ms"
    )

    print(
        f"Median latency : "
        f"{cpu_result['median_latency_ms']:.6f} ms"
    )

    print(
        f"P95 latency    : "
        f"{cpu_result['p95_latency_ms']:.6f} ms"
    )

    print(
        f"FPS            : "
        f"{cpu_result['fps']:.4f}"
    )

    if comparison_rows:

        cpu_row = next(
            (
                row
                for row in comparison_rows
                if row["device"] == "CPU"
            ),
            None
        )

        if cpu_row is not None:

            print(
                f"PyTorch→OpenVINO "
                f"latency speedup: "
                f"{cpu_row['latency_speedup_x']:.4f}x"
            )

            print(
                f"PyTorch→OpenVINO "
                f"FPS speedup: "
                f"{cpu_row['fps_speedup_x']:.4f}x"
            )


# -----------------------------------------------------------------------------
# GPU final summary
# -----------------------------------------------------------------------------

print()

if gpu_result.get("available", False):

    print("OPENVINO GPU")
    print("-" * 50)

    print(
        f"Device         : "
        f"{gpu_result['device']}"
    )

    print(
        f"Mean latency   : "
        f"{gpu_result['mean_latency_ms']:.6f} ms"
    )

    print(
        f"Median latency : "
        f"{gpu_result['median_latency_ms']:.6f} ms"
    )

    print(
        f"P95 latency    : "
        f"{gpu_result['p95_latency_ms']:.6f} ms"
    )

    print(
        f"FPS            : "
        f"{gpu_result['fps']:.4f}"
    )

else:

    print(
        "OpenVINO GPU benchmark:"
    )

    print(
        "  >>> SKIPPED "
        "(no OpenVINO GPU device detected)"
    )


# =============================================================================
# 22. OUTPUT DIRECTORY
# =============================================================================

print()
print("OUTPUT DIRECTORY:")
print(OUTPUT_DIR)

print()
print("Artifacts:")
print(
    "  ",
    comparison_csv
)

print(
    "  ",
    latency_csv
)

print(
    "  ",
    json_path
)

print(
    "  ",
    hash_path
)

print()
print("NEXT:")
print(
    "  >>> Stage 4.9 — "
    "Final CISLR OpenVINO Deployment Evaluation"
)

print()
print("DO NOT RERUN:")
print("  4.3 Computational Benchmark")
print("  4.4 Model Training")
print("  4.5 Official CISLR Evaluation")
print("  4.6 Model Comparison & Selection")
print("  4.7 OpenVINO Conversion")

print()
print("=" * 80)

CISLR — STAGE 4.8 OPENVINO BENCHMARK

Project root:
/home/dipshikha/Music/cao/CISLR_RESEARCH

Selected model:
MobileNetV3-Large

OpenVINO IR:
/home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage4_7_openvino_conversion/fp32/mobilenet_v3_large_stage4_7_fp32.xml

Output:
/home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage4_8_openvino_benchmark

ENVIRONMENT
Python       : 3.11.15 (main, Jun 11 2026, 15:20:16) [GCC 14.3.0]
OpenVINO     : 2026.3.1-22476-759c5a6ab8c-releases/2026/3
Platform     : Linux-7.0.0-31-generic-x86_64-with-glibc2.39
Architecture : x86_64

VERIFYING STAGE 4.7 ARTIFACTS
XML exists : True
BIN exists : True
XML size   : 0.175 MB
BIN size   : 39.263 MB
Total IR   : 39.438 MB

SHA256 XML:
0aed6383ce26dfc4dfbb17eb67f9082ae4ec2fb8c99fac4be7519fce65e60ff0

SHA256 BIN:
02de73f512dd6817c4ef0aafff5d305dc4bd62a9d848aa7084f3f00de8d40a79

OPENVINO CORE
Available OpenVINO devices:
  - CPU
  - GPU

READING OPENVINO FP32 IR
Model read time: 0.009001 seconds

MODEL INFORMATION
Number o

105 warnings generated.
105 warnings generated.
105 warnings generated.
105 warnings generated.
105 warnings generated.
105 warnings generated.
105 warnings generated.
105 warnings generated.


Compilation time: 2.704205 seconds

Warmup: 50 iterations...
Warmup complete.

Measured iterations: 200

RESULTS
--------------------------------------------------
Mean latency     : 3.336119 ms
Median latency   : 3.283241 ms
P95 latency      : 3.740004 ms
P99 latency      : 3.991901 ms
Min latency      : 3.036344 ms
Max latency      : 4.335993 ms
Std latency      : 0.205610 ms
Mean FPS         : 299.7495
Median FPS       : 304.5770

Output shape     : [1, 4764]
Output dtype     : float32

Compiled properties:
  OPTIMAL_NUMBER_OF_INFER_REQUESTS: 1
  NUM_STREAMS: 1
  INFERENCE_NUM_THREADS: None
  PERFORMANCE_HINT: PerformanceMode.LATENCY

STAGE 4.3 PYTORCH vs STAGE 4.8 OPENVINO

CPU
--------------------------------------------------
PyTorch latency : 18.989 ms
OpenVINO latency: 3.702 ms
Latency speedup : 5.1290x
PyTorch FPS     : 52.66
OpenVINO FPS    : 270.10
FPS speedup     : 5.1292x
Latency reduction: 80.50%

OPENVINO GPU
--------------------------------------------------
PyTorch lat